In [ ]:
import os
import sys
import gc
import gzip
import json
import pickle
import random
import collections
import re
import string
import shutil
import math
from collections import Counter, defaultdict
from os.path import join

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import (
    BasicTokenizer,
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    BertTokenizer,
    BertModel,
    BertConfig,
)
from transformers import get_linear_schedule_with_warmup


PRETRAINED_MODEL_PATH = "data/pretrained_model"
RAW_DATA_ROOT = "data/data"
PROCESSED_DATA_ROOT = "data/processed_data"
OUTPUT_ROOT = "output"
EXPERIMENT_NAME = "Exp6-GNN-base"

TRAIN_DATA_PATH = join(RAW_DATA_ROOT, "train.json")
DEV_DATA_PATH = join(RAW_DATA_ROOT, "dev.json")

CUDA_DEVICE_ID = 0
NUM_CPU_THREADS = 25

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
NUM_EPOCHS = 60
GRAD_ACCUM_STEPS = 1
RANDOM_SEED = 42

TYPE_LOSS_WEIGHT = 1.0
SP_LOSS_WEIGHT = 5.0

SP_PROB_THRESHOLD = 0.5

MAX_ANSWER_SPAN_LEN = 30

DOC_WINDOW_OVERLAP_WORDS = 128

ENABLE_SP_THRESHOLD_SEARCH = True
SP_THRESHOLD_COARSE_GRID = np.arange(0.05, 0.951, 0.05)
SP_THRESHOLD_FINE_STEP = 0.01
SP_THRESHOLD_FINE_RADIUS = 0.05

MAX_QUERY_TOKENS = 50
MAX_SEQ_LEN = 512
NUM_ANSWER_TYPES = 4
MAX_SENTENCES = 100

MAX_ENTITIES = 40
MAX_PARAGRAPHS = 10

USE_AMP = False
FORCE_REPROCESS = False
REBUILD_ON_INVALID_FEATURES = True

ENABLE_EARLY_STOPPING = True
EARLY_STOP_METRIC = "joint_f1"
EARLY_STOP_MODE = "max"
EARLY_STOP_PATIENCE = 30
EARLY_STOP_MIN_DELTA = 1e-6

TQDM_KW = dict(ncols=120, leave=True, dynamic_ncols=True, mininterval=0.5)
TRAIN_TQDM_KW = dict(ncols=120, leave=True, dynamic_ncols=True, mininterval=0.5)
EVAL_TQDM_KW = dict(ncols=120, leave=True, dynamic_ncols=True, mininterval=0.5)

IGNORE_INDEX = -100


os.environ["OMP_NUM_THREADS"] = str(NUM_CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(NUM_CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

torch.set_num_threads(NUM_CPU_THREADS)
try:
    torch.set_num_interop_threads(min(4, NUM_CPU_THREADS))
except Exception:
    pass

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _local_only(path_or_name: str) -> bool:
    return os.path.isdir(path_or_name)


def _autocast(enabled: bool):
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast("cuda", enabled=enabled)
    from torch.cuda.amp import autocast

    return autocast(enabled=enabled)


def _grad_scaler(enabled: bool):
    if not enabled:
        return None
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            return torch.amp.GradScaler("cuda")
        except TypeError:
            return torch.amp.GradScaler(device_type="cuda")
    from torch.cuda.amp import GradScaler

    return GradScaler()


def _atomic_write_json(path, obj):
    tmp_path = path + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=4, ensure_ascii=False)
    os.replace(tmp_path, path)


def _tqdm_write(msg: str):
    try:
        tqdm.write(str(msg))
    except Exception:
        sys.stdout.write(str(msg) + "\n")
        sys.stdout.flush()


def _auto_error_missing_model_type(err: Exception) -> bool:
    msg = str(err)
    return ("model_type" in msg) and ("Unrecognized model" in msg)


def _load_tokenizer(model_path: str, local_only: bool):
    try:
        try:
            return AutoTokenizer.from_pretrained(
                model_path, local_files_only=local_only, use_fast=True
            )
        except Exception:
            return AutoTokenizer.from_pretrained(
                model_path, local_files_only=local_only, use_fast=False
            )
    except ValueError as e:
        if _auto_error_missing_model_type(e):
            return BertTokenizer.from_pretrained(
                model_path, local_files_only=local_only
            )
        raise


def _load_config_and_encoder(model_path: str, local_only: bool):
    try:
        config = AutoConfig.from_pretrained(model_path, local_files_only=local_only)
        encoder = AutoModel.from_pretrained(model_path, local_files_only=local_only)
        return config, encoder
    except ValueError as e:
        if _auto_error_missing_model_type(e):
            config = BertConfig.from_pretrained(model_path, local_files_only=local_only)
            encoder = BertModel.from_pretrained(model_path, local_files_only=local_only)
            return config, encoder
        raise


def _get_cls_sep(tokenizer):
    cls_tok = getattr(tokenizer, "cls_token", None) or "[CLS]"
    sep_tok = getattr(tokenizer, "sep_token", None) or "[SEP]"
    return cls_tok, sep_tok


def _strip_spaces_with_map(text: str):
    ns_chars = []
    ns_to_s = []
    for i, c in enumerate(text):
        if c == " ":
            continue
        ns_to_s.append(i)
        ns_chars.append(c)
    return "".join(ns_chars), ns_to_s


def _find_answer_char_spans(sent_text: str, answer_text: str):
    answer_text = (answer_text or "").strip()
    if not answer_text:
        return []

    spans = []
    offset = -1
    while True:
        offset = sent_text.find(answer_text, offset + 1)
        if offset == -1:
            break
        spans.append((offset, offset + len(answer_text) - 1))
    if spans:
        return spans

    sent_ns, ns_to_s = _strip_spaces_with_map(sent_text)
    ans_ns = answer_text.replace(" ", "")
    if not ans_ns:
        return []

    ns_offset = -1
    while True:
        ns_offset = sent_ns.find(ans_ns, ns_offset + 1)
        if ns_offset == -1:
            break
        ns_start = ns_offset
        ns_end = ns_offset + len(ans_ns) - 1
        if 0 <= ns_start < len(ns_to_s) and 0 <= ns_end < len(ns_to_s):
            spans.append((ns_to_s[ns_start], ns_to_s[ns_end]))
    return spans


def _is_whitespace(c: str) -> bool:
    return c in (" ", "\t", "\r", "\n") or ord(c) == 0x202F


def _append_sentence_as_char_tokens(
    sent_text: str, doc_tokens: list, char_to_word_offset: list
):
    for c in sent_text:
        if _is_whitespace(c):
            char_to_word_offset.append(len(doc_tokens) - 1)
            continue
        doc_tokens.append(c)
        char_to_word_offset.append(len(doc_tokens) - 1)


class EarlyStopping:
    def __init__(self, patience: int, mode: str = "max", min_delta: float = 0.0):
        if patience < 1:
            raise ValueError("patience must be >= 1")
        if mode not in ("max", "min"):
            raise ValueError("mode must be 'max' or 'min'")
        self.patience = int(patience)
        self.mode = mode
        self.min_delta = float(min_delta)
        self.best_score = None
        self.num_bad_epochs = 0

    def _is_improved(self, score: float) -> bool:
        if self.best_score is None:
            return True
        if self.mode == "max":
            return score > (self.best_score + self.min_delta)
        return score < (self.best_score - self.min_delta)

    def step(self, score: float):
        score = float(score)
        improved = self._is_improved(score)
        if improved:
            self.best_score = score
            self.num_bad_epochs = 0
        else:
            self.num_bad_epochs += 1
        should_stop = self.num_bad_epochs >= self.patience
        return improved, should_stop


class CheckpointManager:
    def __init__(self, output_dir: str, exp_name: str):
        self.ckpt_dir = join(output_dir, "checkpoints", exp_name)
        self.sub_dir = join(output_dir, "submissions", exp_name)
        os.makedirs(self.ckpt_dir, exist_ok=True)
        os.makedirs(self.sub_dir, exist_ok=True)

        self.last_ckpt_path = join(self.ckpt_dir, "last.pth")
        self.best_ckpt_path = join(self.ckpt_dir, "best.pth")

        self.last_pred_path = join(self.sub_dir, "last.json")
        self.best_pred_path = join(self.sub_dir, "best.json")

        self.last_metrics_path = join(self.sub_dir, "last_metrics.json")
        self.best_metrics_path = join(self.sub_dir, "best_metrics.json")

        self.tmp_pred_path = join(self.sub_dir, "tmp_pred.json")

    def save_last_ckpt(self, model_state_dict):
        torch.save(model_state_dict, self.last_ckpt_path)

    def save_best_ckpt_from_last(self):
        shutil.copyfile(self.last_ckpt_path, self.best_ckpt_path)

    def finalize_last_pred_from_tmp(self):
        os.replace(self.tmp_pred_path, self.last_pred_path)

    def save_best_pred_from_last(self):
        shutil.copyfile(self.last_pred_path, self.best_pred_path)

    def save_last_metrics(self, metrics_obj):
        _atomic_write_json(self.last_metrics_path, metrics_obj)

    def save_best_metrics(self):
        shutil.copyfile(self.last_metrics_path, self.best_metrics_path)


class Example(object):
    def __init__(
        self,
        qas_id,
        qas_type,
        doc_tokens,
        question_text,
        sent_num,
        sent_names,
        sup_fact_id,
        para_start_end_position,
        sent_start_end_position,
        entity_start_end_position,
        orig_answer_text=None,
        start_position=None,
        end_position=None,
    ):
        self.qas_id = qas_id
        self.qas_type = qas_type
        self.doc_tokens = doc_tokens
        self.question_text = question_text
        self.sent_num = sent_num
        self.sent_names = sent_names
        self.sup_fact_id = sup_fact_id
        self.para_start_end_position = para_start_end_position
        self.sent_start_end_position = sent_start_end_position
        self.entity_start_end_position = entity_start_end_position
        self.orig_answer_text = orig_answer_text
        self.start_position = start_position
        self.end_position = end_position


class InputFeatures(object):
    def __init__(
        self,
        qas_id,
        doc_tokens,
        doc_input_ids,
        doc_input_mask,
        doc_segment_ids,
        query_tokens,
        query_input_ids,
        query_input_mask,
        query_segment_ids,
        sent_spans,
        sent_names,
        sup_fact_ids,
        ans_type,
        token_to_orig_map,
        entity_spans=None,
        para_spans=None,
        start_position=None,
        end_position=None,
    ):
        self.qas_id = qas_id
        self.doc_tokens = doc_tokens
        self.doc_input_ids = doc_input_ids
        self.doc_input_mask = doc_input_mask
        self.doc_segment_ids = doc_segment_ids
        self.query_tokens = query_tokens
        self.query_input_ids = query_input_ids
        self.query_input_mask = query_input_mask
        self.query_segment_ids = query_segment_ids
        self.sent_spans = sent_spans
        self.sent_names = sent_names
        self.sup_fact_ids = sup_fact_ids
        self.ans_type = ans_type
        self.token_to_orig_map = token_to_orig_map
        self.entity_spans = entity_spans if entity_spans is not None else []
        self.para_spans = para_spans if para_spans is not None else []
        self.start_position = start_position
        self.end_position = end_position


class _CompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if name == "Example":
            return Example
        if name == "InputFeatures":
            return InputFeatures
        return super().find_class(module, name)


def _load_pickle(path):
    with gzip.open(path, "rb") as f:
        return _CompatUnpickler(f).load()


def _dump_pickle(path, obj):
    with gzip.open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


def read_examples(full_file):
    with open(full_file, "r", encoding="utf-8") as reader:
        full_data = json.load(reader)

    examples = []
    for case in tqdm(
        full_data, desc=f"read_examples:{os.path.basename(full_file)}", **TQDM_KW
    ):
        qas_id = case["_id"]
        qas_type = ""

        sup_facts = set((sp[0], sp[1]) for sp in case.get("supporting_facts", []))
        sup_titles = set(sp[0] for sp in case.get("supporting_facts", []))

        orig_answer_text = case.get("answer", "")
        if not isinstance(orig_answer_text, str):
            orig_answer_text = str(orig_answer_text)
        orig_answer_text = orig_answer_text.strip()

        sent_id = 0
        doc_tokens = []
        sent_names = []
        sup_facts_sent_names = []
        sent_start_end_position = []
        para_start_end_position = []
        entity_start_end_position = []
        ans_start_position, ans_end_position = [], []

        is_judge_answer = orig_answer_text in ("yes", "no", "unknown", "")
        found_span = False

        char_to_word_offset = []

        para_data = case.get("context", [])
        for paragraph in para_data:
            if not isinstance(paragraph, list) or len(paragraph) < 2:
                continue
            title = paragraph[0]
            sents = paragraph[1]
            if not isinstance(sents, list):
                continue

            is_gold_para = 1 if title in sup_titles else 0
            para_start_position = len(doc_tokens)

            for local_sent_id, sent in enumerate(sents):
                if local_sent_id >= 100:
                    break
                if not isinstance(sent, str):
                    sent = str(sent)

                local_sent_name = (title, local_sent_id)
                sent_names.append(local_sent_name)
                if local_sent_name in sup_facts:
                    sup_facts_sent_names.append(local_sent_name)
                sent_id += 1

                sent_text = sent + " "
                sent_start_char_id = len(char_to_word_offset)

                sent_token_start = len(doc_tokens)
                _append_sentence_as_char_tokens(
                    sent_text, doc_tokens, char_to_word_offset
                )
                sent_token_end = len(doc_tokens) - 1
                if sent_token_end >= sent_token_start:
                    sent_start_end_position.append((sent_token_start, sent_token_end))
                else:
                    sent_start_end_position.append(
                        (sent_token_start, sent_token_start - 1)
                    )

                if (not is_judge_answer) and (not found_span):
                    spans = _find_answer_char_spans(sent_text, orig_answer_text)
                    if spans:
                        found_span = True
                        for s_off, e_off in spans:
                            start_char_position = sent_start_char_id + s_off
                            end_char_position = sent_start_char_id + e_off
                            if 0 <= start_char_position < len(
                                char_to_word_offset
                            ) and 0 <= end_char_position < len(char_to_word_offset):
                                s_tok = char_to_word_offset[start_char_position]
                                e_tok = char_to_word_offset[end_char_position]
                                if s_tok != -1 and e_tok != -1 and s_tok <= e_tok:
                                    ans_start_position.append(s_tok)
                                    ans_end_position.append(e_tok)

            para_end_position = len(doc_tokens) - 1
            para_start_end_position.append(
                (para_start_position, para_end_position, title, is_gold_para)
            )

        examples.append(
            Example(
                qas_id=qas_id,
                qas_type=qas_type,
                doc_tokens=doc_tokens,
                question_text=case.get("question", ""),
                sent_num=sent_id,
                sent_names=sent_names,
                sup_fact_id=sup_facts_sent_names,
                para_start_end_position=para_start_end_position,
                sent_start_end_position=sent_start_end_position,
                entity_start_end_position=entity_start_end_position,
                orig_answer_text=orig_answer_text,
                start_position=ans_start_position,
                end_position=ans_end_position,
            )
        )

    return examples


def get_valid_spans(spans, limit):
    new_spans = []
    for span in spans:
        if span[1] < limit:
            new_spans.append(span)
        else:
            new_span = list(span)
            new_span[1] = limit - 1
            new_spans.append(tuple(new_span))
            break
    return new_spans


def get_valid_spans_with_names(spans, names, limit):
    new_spans = []
    new_names = []
    for span, name in zip(spans, names):
        if span[1] < limit:
            new_spans.append(span)
            new_names.append(name)
        else:
            new_span = list(span)
            new_span[1] = limit - 1
            new_spans.append(tuple(new_span))
            new_names.append(name)
            break
    return new_spans, new_names


def _improve_answer_span(
    doc_tokens, input_start, input_end, tokenizer, orig_answer_text
):
    tok_answer_text = " ".join(tokenizer.tokenize(orig_answer_text))
    for new_start in range(input_start, input_end + 1):
        for new_end in range(input_end, new_start - 1, -1):
            text_span = " ".join(doc_tokens[new_start : (new_end + 1)])
            if text_span == tok_answer_text:
                return new_start, new_end
    return input_start, input_end


def _build_wordpiece_cache(doc_tokens, tokenizer):
    pieces = []
    lens = []
    unk = getattr(tokenizer, "unk_token", "[UNK]")
    for w in doc_tokens:
        p = tokenizer.tokenize(w)
        if not p:
            p = [unk]
        pieces.append(p)
        lens.append(len(p))
    return pieces, lens


def _iter_windows(wordpiece_lens, max_tokens_for_doc):
    doc_len = len(wordpiece_lens)
    start = 0
    while start < doc_len:
        end = start
        total = 0
        while end < doc_len:
            l = wordpiece_lens[end] if wordpiece_lens[end] > 0 else 1
            if total + l > max_tokens_for_doc:
                break
            total += l
            end += 1
        if end == start:
            end = min(start + 1, doc_len)

        yield start, end

        if end >= doc_len:
            break
        window_words = end - start
        stride = max(1, window_words - DOC_WINDOW_OVERLAP_WORDS)
        start += stride


_ENTITY_PATTERNS = [
    re.compile(r"\d+[\.,]?\d*[元万亿]"),
    re.compile(r"\d{4}年\d{1,2}月\d{1,2}日"),
    re.compile(r"[\u4e00-\u9fa5]{2,10}(局|处|院|公司|行|社|所|部|委|室|站)"),
]


def _extract_simple_entities(doc_tokens):
    text = "".join(doc_tokens)
    spans = []
    for pat in _ENTITY_PATTERNS:
        for m in pat.finditer(text):
            s, e = m.span()
            if e - s <= 1:
                continue
            spans.append((s, e - 1))

    uniq = []
    seen = set()
    for s, e in spans:
        s = max(0, s)
        e = min(len(doc_tokens) - 1, e)
        if s > e:
            continue
        surface = text[s : e + 1]
        if surface in seen:
            continue
        seen.add(surface)
        uniq.append((s, e))

    return uniq[:MAX_ENTITIES]


def convert_examples_to_features(examples, tokenizer, max_seq_length, max_query_length):
    cls_tok, sep_tok = _get_cls_sep(tokenizer)
    use_token_type_ids = "token_type_ids" in getattr(tokenizer, "model_input_names", [])

    features = []
    for example in tqdm(examples, desc="convert_examples_to_features", **TQDM_KW):
        if example.orig_answer_text == "yes":
            ans_type = 1
        elif example.orig_answer_text == "no":
            ans_type = 2
        elif example.orig_answer_text == "unknown":
            ans_type = 3
        else:
            ans_type = 0

        q_pieces = tokenizer.tokenize(example.question_text or "")
        q_pieces = q_pieces[: max(0, max_query_length - 2)]
        query_tokens = [cls_tok] + q_pieces + [sep_tok]

        doc_wordpieces, doc_wordpiece_lens = _build_wordpiece_cache(
            example.doc_tokens, tokenizer
        )

        max_tokens_for_doc = max_seq_length - len(query_tokens) - 1
        if max_tokens_for_doc < 1:
            max_tokens_for_doc = 1

        for win_start, win_end in _iter_windows(doc_wordpiece_lens, max_tokens_for_doc):
            all_doc_tokens = list(query_tokens)
            tok_to_orig_index = [0] * len(query_tokens)

            word_to_tok_start = {}
            word_to_tok_end = {}

            for orig_i in range(win_start, win_end):
                word_to_tok_start[orig_i] = len(all_doc_tokens)
                for st in doc_wordpieces[orig_i]:
                    tok_to_orig_index.append(orig_i)
                    all_doc_tokens.append(st)
                word_to_tok_end[orig_i] = len(all_doc_tokens) - 1

            all_doc_tokens = all_doc_tokens[: max_seq_length - 1] + [sep_tok]
            doc_input_ids = tokenizer.convert_tokens_to_ids(all_doc_tokens)
            query_input_ids = tokenizer.convert_tokens_to_ids(query_tokens)

            doc_input_mask = [1] * len(doc_input_ids)
            if use_token_type_ids:
                doc_segment_ids = [0] * len(query_input_ids) + [1] * (
                    len(doc_input_ids) - len(query_input_ids)
                )
            else:
                doc_segment_ids = [0] * len(doc_input_ids)

            while len(doc_input_ids) < max_seq_length:
                doc_input_ids.append(0)
                doc_input_mask.append(0)
                doc_segment_ids.append(0)

            query_input_mask = [1] * len(query_input_ids)
            query_segment_ids = [0] * len(query_input_ids)
            while len(query_input_ids) < max_query_length:
                query_input_ids.append(0)
                query_input_mask.append(0)
                query_segment_ids.append(0)

            sent_spans = []
            sent_names_local = []
            for global_sid, (s_w, e_w) in enumerate(example.sent_start_end_position):
                if e_w < win_start or s_w >= win_end:
                    continue
                s2 = max(s_w, win_start)
                e2 = min(e_w, win_end - 1)
                if s2 > e2:
                    continue
                if s2 not in word_to_tok_start or e2 not in word_to_tok_end:
                    continue
                tok_s = word_to_tok_start[s2]
                tok_e = word_to_tok_end[e2]
                if tok_s > tok_e:
                    continue
                sent_spans.append((tok_s, tok_e))
                if global_sid < len(example.sent_names):
                    sent_names_local.append(example.sent_names[global_sid])
                else:
                    sent_names_local.append(("", global_sid))

            sent_spans, sent_names_local = get_valid_spans_with_names(
                sent_spans, sent_names_local, max_seq_length
            )

            sent_spans = sent_spans[:MAX_SENTENCES]
            sent_names_local = sent_names_local[:MAX_SENTENCES]

            if len(sent_spans) == 0:
                doc_start = min(len(query_tokens), max_seq_length - 1)
                doc_end = max(
                    doc_start, min(len(all_doc_tokens) - 2, max_seq_length - 2)
                )
                sent_spans = [(doc_start, doc_end)]
                sent_names_local = [("", 0)]

            para_spans = []
            for p_start, p_end, p_title, is_gold in example.para_start_end_position:
                if p_end < win_start or p_start >= win_end:
                    continue
                s2 = max(p_start, win_start)
                e2 = min(p_end, win_end - 1)
                if s2 > e2:
                    continue
                if s2 not in word_to_tok_start or e2 not in word_to_tok_end:
                    continue
                tok_s = word_to_tok_start[s2]
                tok_e = word_to_tok_end[e2]
                if tok_s > tok_e:
                    continue
                para_spans.append((tok_s, tok_e))

            para_spans = get_valid_spans(para_spans, max_seq_length)[:MAX_PARAGRAPHS]

            window_doc_tokens = example.doc_tokens[win_start:win_end]
            raw_entity_spans = _extract_simple_entities(window_doc_tokens)

            entity_spans = []
            for rel_s, rel_e in raw_entity_spans:
                abs_s = win_start + rel_s
                abs_e = win_start + rel_e
                if abs_s not in word_to_tok_start or abs_e not in word_to_tok_end:
                    continue
                tok_s = word_to_tok_start[abs_s]
                tok_e = word_to_tok_end[abs_e]
                if tok_s > tok_e:
                    continue
                entity_spans.append((tok_s, tok_e))

            entity_spans = get_valid_spans(entity_spans, max_seq_length)[:MAX_ENTITIES]

            sup_fact_ids = []
            sup_set = set(example.sup_fact_id)
            for local_j, name in enumerate(sent_names_local):
                if name in sup_set:
                    sup_fact_ids.append(local_j)

            ans_start_position, ans_end_position = [], []
            if ans_type == 0 and example.start_position and example.end_position:
                for a_s, a_e in zip(example.start_position, example.end_position):
                    if a_s is None or a_e is None:
                        continue
                    if a_s < win_start or a_e >= win_end:
                        continue
                    if a_s not in word_to_tok_start or a_e not in word_to_tok_end:
                        continue
                    tok_s = word_to_tok_start[a_s]
                    tok_e = word_to_tok_end[a_e]
                    tok_s, tok_e = _improve_answer_span(
                        all_doc_tokens,
                        tok_s,
                        tok_e,
                        tokenizer,
                        example.orig_answer_text,
                    )
                    if (
                        0 <= tok_s < max_seq_length
                        and 0 <= tok_e < max_seq_length
                        and tok_s <= tok_e
                    ):
                        ans_start_position.append(tok_s)
                        ans_end_position.append(tok_e)

            features.append(
                InputFeatures(
                    qas_id=example.qas_id,
                    doc_tokens=all_doc_tokens,
                    doc_input_ids=doc_input_ids,
                    doc_input_mask=doc_input_mask,
                    doc_segment_ids=doc_segment_ids,
                    query_tokens=query_tokens,
                    query_input_ids=query_input_ids,
                    query_input_mask=query_input_mask,
                    query_segment_ids=query_segment_ids,
                    sent_spans=sent_spans,
                    sent_names=sent_names_local,
                    sup_fact_ids=sup_fact_ids,
                    ans_type=ans_type,
                    token_to_orig_map=tok_to_orig_index,
                    entity_spans=entity_spans,
                    para_spans=para_spans,
                    start_position=ans_start_position,
                    end_position=ans_end_position,
                )
            )

    return features


def _feature_ok(path):
    if not os.path.exists(path):
        return False
    try:
        feats = _load_pickle(path)
        if not feats:
            return False

        for x in feats[: min(50, len(feats))]:
            if (
                not hasattr(x, "sent_spans")
                or not hasattr(x, "sent_names")
                or not hasattr(x, "sup_fact_ids")
            ):
                return False
            if not hasattr(x, "entity_spans") or not hasattr(x, "para_spans"):
                return False

        return any(getattr(x, "sent_spans", []) for x in feats[: min(200, len(feats))])
    except Exception:
        return False


def ensure_preprocessed():
    os.makedirs(PROCESSED_DATA_ROOT, exist_ok=True)
    train_example_path = join(PROCESSED_DATA_ROOT, "train_example.pkl.gz")
    train_feature_path = join(PROCESSED_DATA_ROOT, "train_feature.pkl.gz")
    dev_example_path = join(PROCESSED_DATA_ROOT, "dev_example.pkl.gz")
    dev_feature_path = join(PROCESSED_DATA_ROOT, "dev_feature.pkl.gz")

    need = FORCE_REPROCESS
    if not need:
        need = not (
            os.path.exists(train_example_path)
            and os.path.exists(dev_example_path)
            and os.path.exists(train_feature_path)
            and os.path.exists(dev_feature_path)
        )
    if not need and REBUILD_ON_INVALID_FEATURES:
        need = not (_feature_ok(train_feature_path) and _feature_ok(dev_feature_path))

    if not need:
        return

    local_only = _local_only(PRETRAINED_MODEL_PATH)
    tokenizer = _load_tokenizer(PRETRAINED_MODEL_PATH, local_only=local_only)

    train_examples = read_examples(TRAIN_DATA_PATH)
    _dump_pickle(train_example_path, train_examples)
    train_features = convert_examples_to_features(
        train_examples,
        tokenizer,
        max_seq_length=MAX_SEQ_LEN,
        max_query_length=MAX_QUERY_TOKENS,
    )
    _dump_pickle(train_feature_path, train_features)
    del train_examples, train_features
    gc.collect()

    dev_examples = read_examples(DEV_DATA_PATH)
    _dump_pickle(dev_example_path, dev_examples)
    dev_features = convert_examples_to_features(
        dev_examples,
        tokenizer,
        max_seq_length=MAX_SEQ_LEN,
        max_query_length=MAX_QUERY_TOKENS,
    )
    _dump_pickle(dev_feature_path, dev_features)
    del dev_examples, dev_features
    gc.collect()


class DataIteratorPack(object):
    def __init__(
        self, features, example_dict, bsz, device, sent_limit, sequential=False
    ):
        self.bsz = bsz
        self.device = device
        self.features = features
        self.example_dict = example_dict
        self.sequential = sequential
        self.sent_limit = sent_limit
        self.example_ptr = 0
        if not sequential:
            np.random.shuffle(self.features)

    def refresh(self):
        self.example_ptr = 0
        if not self.sequential:
            np.random.shuffle(self.features)

    def empty(self):
        return self.example_ptr >= len(self.features)

    def __len__(self):
        return int(np.ceil(len(self.features) / self.bsz))

    def __iter__(self):
        context_idxs = torch.LongTensor(self.bsz, MAX_SEQ_LEN).cuda(self.device)
        context_mask = torch.LongTensor(self.bsz, MAX_SEQ_LEN).cuda(self.device)
        segment_idxs = torch.LongTensor(self.bsz, MAX_SEQ_LEN).cuda(self.device)

        query_mapping = torch.Tensor(self.bsz, MAX_SEQ_LEN).cuda(self.device)

        sent_start_mapping = torch.Tensor(self.bsz, self.sent_limit, MAX_SEQ_LEN).cuda(
            self.device
        )
        sent_all_mapping = torch.Tensor(self.bsz, MAX_SEQ_LEN, self.sent_limit).cuda(
            self.device
        )

        entity_mapping = torch.Tensor(self.bsz, MAX_SEQ_LEN, MAX_ENTITIES).cuda(
            self.device
        )
        para_mapping = torch.Tensor(self.bsz, MAX_SEQ_LEN, MAX_PARAGRAPHS).cuda(
            self.device
        )

        y1 = torch.LongTensor(self.bsz).cuda(self.device)
        y2 = torch.LongTensor(self.bsz).cuda(self.device)
        q_type = torch.LongTensor(self.bsz).cuda(self.device)
        is_support = torch.FloatTensor(self.bsz, self.sent_limit).cuda(self.device)

        while True:
            if self.example_ptr >= len(self.features):
                break

            start_id = self.example_ptr
            cur_bsz = min(self.bsz, len(self.features) - start_id)
            cur_batch = self.features[start_id : start_id + cur_bsz]
            cur_batch.sort(key=lambda x: sum(x.doc_input_mask), reverse=True)

            ids = []
            max_sent_cnt = 0
            max_ent_cnt = 0
            max_para_cnt = 0

            for mapping in [
                sent_start_mapping,
                sent_all_mapping,
                query_mapping,
                entity_mapping,
                para_mapping,
            ]:
                mapping.zero_()
            is_support.fill_(0)

            for i in range(len(cur_batch)):
                case = cur_batch[i]

                context_idxs[i].copy_(
                    torch.as_tensor(case.doc_input_ids, device=context_idxs.device)
                )
                context_mask[i].copy_(
                    torch.as_tensor(case.doc_input_mask, device=context_mask.device)
                )
                segment_idxs[i].copy_(
                    torch.as_tensor(case.doc_segment_ids, device=segment_idxs.device)
                )

                if case.sent_spans:
                    q_end = max(case.sent_spans[0][0] - 1, 0)
                    if q_end > 0:
                        query_mapping[i, :q_end] = 1

                if case.ans_type == 0:
                    if len(case.end_position) == 0:
                        y1[i] = IGNORE_INDEX
                        y2[i] = IGNORE_INDEX
                    elif case.end_position[0] < MAX_SEQ_LEN:
                        y1[i] = case.start_position[0]
                        y2[i] = case.end_position[0]
                    else:
                        y1[i] = IGNORE_INDEX
                        y2[i] = IGNORE_INDEX
                    q_type[i] = 0
                elif case.ans_type == 1:
                    y1[i] = IGNORE_INDEX
                    y2[i] = IGNORE_INDEX
                    q_type[i] = 1
                elif case.ans_type == 2:
                    y1[i] = IGNORE_INDEX
                    y2[i] = IGNORE_INDEX
                    q_type[i] = 2
                elif case.ans_type == 3:
                    y1[i] = IGNORE_INDEX
                    y2[i] = IGNORE_INDEX
                    q_type[i] = 3

                for j, sent_span in enumerate(case.sent_spans[: self.sent_limit]):
                    is_sp_flag = j in case.sup_fact_ids
                    start, end = sent_span
                    if start <= end:
                        is_support[i, j] = int(is_sp_flag)
                        sent_all_mapping[i, start : end + 1, j] = 1
                        sent_start_mapping[i, j, start] = 1

                max_sent_cnt = max(
                    max_sent_cnt, min(len(case.sent_spans), self.sent_limit)
                )

                for j, ent_span in enumerate(case.entity_spans[:MAX_ENTITIES]):
                    start, end = ent_span
                    if start <= end:
                        entity_mapping[i, start : end + 1, j] = 1
                max_ent_cnt = max(
                    max_ent_cnt, min(len(case.entity_spans), MAX_ENTITIES)
                )

                for j, para_span in enumerate(case.para_spans[:MAX_PARAGRAPHS]):
                    start, end = para_span
                    if start <= end:
                        para_mapping[i, start : end + 1, j] = 1
                max_para_cnt = max(
                    max_para_cnt, min(len(case.para_spans), MAX_PARAGRAPHS)
                )

                ids.append(case.qas_id)

            input_lengths = (context_mask[:cur_bsz] > 0).long().sum(dim=1)
            max_c_len = int(input_lengths.max())

            self.example_ptr += cur_bsz

            max_sent_cnt = max(1, max_sent_cnt)
            max_ent_cnt = max(1, max_ent_cnt)
            max_para_cnt = max(1, max_para_cnt)

            yield {
                "context_idxs": context_idxs[:cur_bsz, :max_c_len].contiguous(),
                "context_mask": context_mask[:cur_bsz, :max_c_len].contiguous(),
                "segment_idxs": segment_idxs[:cur_bsz, :max_c_len].contiguous(),
                "query_mapping": query_mapping[:cur_bsz, :max_c_len].contiguous(),
                "y1": y1[:cur_bsz],
                "y2": y2[:cur_bsz],
                "ids": ids,
                "q_type": q_type[:cur_bsz],
                "sent_start_mapping": sent_start_mapping[
                    :cur_bsz, :max_sent_cnt, :max_c_len
                ],
                "sent_all_mapping": sent_all_mapping[
                    :cur_bsz, :max_c_len, :max_sent_cnt
                ],
                "entity_mapping": entity_mapping[:cur_bsz, :max_c_len, :max_ent_cnt],
                "para_mapping": para_mapping[:cur_bsz, :max_c_len, :max_para_cnt],
                "is_support": is_support[:cur_bsz, :max_sent_cnt].contiguous(),
                "features": cur_batch,
            }


class DataHelper:
    def __init__(self, data_dir, gz=True):
        self.gz = gz
        self.suffix = ".pkl.gz" if gz else ".pkl"
        self.data_dir = data_dir
        self.__train_features__ = None
        self.__dev_features__ = None
        self.__train_examples__ = None
        self.__dev_examples__ = None
        self.__train_example_dict__ = None
        self.__dev_example_dict__ = None

    def _feature_file(self, tag):
        return join(self.data_dir, tag + "_feature" + self.suffix)

    def _example_file(self, tag):
        return join(self.data_dir, tag + "_example" + self.suffix)

    @property
    def train_feature_file(self):
        return self._feature_file("train")

    @property
    def dev_feature_file(self):
        return self._feature_file("dev")

    @property
    def train_example_file(self):
        return self._example_file("train")

    @property
    def dev_example_file(self):
        return self._example_file("dev")

    def _get_or_load(self, attr, path):
        v = getattr(self, attr)
        if v is None:
            _tqdm_write(f"loading {path}")
            setattr(self, attr, _load_pickle(path))
        return getattr(self, attr)

    @property
    def train_features(self):
        return self._get_or_load("__train_features__", self.train_feature_file)

    @property
    def dev_features(self):
        return self._get_or_load("__dev_features__", self.dev_feature_file)

    @property
    def train_examples(self):
        return self._get_or_load("__train_examples__", self.train_example_file)

    @property
    def dev_examples(self):
        return self._get_or_load("__dev_examples__", self.dev_example_file)

    @property
    def train_example_dict(self):
        if self.__train_example_dict__ is None:
            self.__train_example_dict__ = {e.qas_id: e for e in self.train_examples}
        return self.__train_example_dict__

    @property
    def dev_example_dict(self):
        if self.__dev_example_dict__ is None:
            self.__dev_example_dict__ = {e.qas_id: e for e in self.dev_examples}
        return self.__dev_example_dict__

    def train_loader(self, bsz, device, sent_limit):
        return DataIteratorPack(
            self.train_features,
            self.train_example_dict,
            bsz=bsz,
            device=device,
            sent_limit=sent_limit,
            sequential=False,
        )

    def dev_loader(self, bsz, device, sent_limit):
        return DataIteratorPack(
            self.dev_features,
            self.dev_example_dict,
            bsz=bsz,
            device=device,
            sent_limit=sent_limit,
            sequential=True,
        )


class HeterogeneousGraphLayer(nn.Module):
    def __init__(self, input_dim, dropout=0.1):
        super(HeterogeneousGraphLayer, self).__init__()
        self.input_dim = input_dim

        self.ss_attn = nn.MultiheadAttention(input_dim, num_heads=1, batch_first=True)
        self.es_attn = nn.MultiheadAttention(input_dim, num_heads=1, batch_first=True)
        self.ps_attn = nn.MultiheadAttention(input_dim, num_heads=1, batch_first=True)

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(input_dim)

    def _safe_cross_attn(self, attn, query, key, key_padding_mask):
        B = query.size(0)
        out = torch.zeros_like(query)

        if key.size(1) == 0:
            return out

        if key_padding_mask is None:
            vmask = torch.ones(B, device=query.device, dtype=torch.bool)
        else:
            vmask = (~key_padding_mask).any(dim=1)

        if not vmask.any():
            return out

        q2 = query[vmask]
        k2 = key[vmask]
        m2 = key_padding_mask[vmask] if key_padding_mask is not None else None
        o2, _ = attn(query=q2, key=k2, value=k2, key_padding_mask=m2)
        out[vmask] = o2
        return out

    def forward(
        self,
        sent_nodes,
        ent_nodes,
        para_nodes,
        sent_mask=None,
        ent_mask=None,
        para_mask=None,
    ):
        ss_out, _ = self.ss_attn(
            sent_nodes, sent_nodes, sent_nodes, key_padding_mask=sent_mask
        )

        es_out = self._safe_cross_attn(self.es_attn, sent_nodes, ent_nodes, ent_mask)
        ps_out = self._safe_cross_attn(self.ps_attn, sent_nodes, para_nodes, para_mask)

        merged = ss_out + es_out + ps_out
        out = sent_nodes + self.dropout(merged)
        out = self.norm(out)
        return out


class SimplePredictionLayer(nn.Module):
    def __init__(self, input_dim, label_type_num, max_answer_len=MAX_ANSWER_SPAN_LEN):
        super(SimplePredictionLayer, self).__init__()
        self.input_dim = input_dim
        self.max_answer_len = int(max_answer_len)

        self.hetero_gnn = HeterogeneousGraphLayer(input_dim)

        self.sp_linear = nn.Linear(self.input_dim, 1)
        self.start_linear = nn.Linear(self.input_dim, 1)
        self.end_linear = nn.Linear(self.input_dim, 1)
        self.type_linear = nn.Linear(self.input_dim, label_type_num)
        self.cache_S = 0
        self.cache_mask = None

    def get_output_mask(self, outer):
        S = outer.size(1)
        if S <= self.cache_S:
            return self.cache_mask[:S, :S]
        self.cache_S = S
        np_mask = np.tril(np.triu(np.ones((S, S)), 0), self.max_answer_len)
        self.cache_mask = outer.data.new(S, S).copy_(torch.from_numpy(np_mask))
        return self.cache_mask

    def forward(self, batch, input_state):
        query_mapping = batch["query_mapping"]
        context_mask = batch["context_mask"]

        sent_all_mapping = batch["sent_all_mapping"]
        entity_mapping = batch["entity_mapping"]
        para_mapping = batch["para_mapping"]

        s_state = sent_all_mapping.unsqueeze(3) * input_state.unsqueeze(2)
        sent_nodes = s_state.max(1)[0]

        e_state = entity_mapping.unsqueeze(3) * input_state.unsqueeze(2)
        ent_nodes = e_state.max(1)[0]

        p_state = para_mapping.unsqueeze(3) * input_state.unsqueeze(2)
        para_nodes = p_state.max(1)[0]

        sent_pad_mask = ~(sent_all_mapping.sum(1) > 0)
        ent_pad_mask = ~(entity_mapping.sum(1) > 0)
        para_pad_mask = ~(para_mapping.sum(1) > 0)

        enhanced_sent_nodes = self.hetero_gnn(
            sent_nodes,
            ent_nodes,
            para_nodes,
            sent_mask=sent_pad_mask,
            ent_mask=ent_pad_mask,
            para_mask=para_pad_mask,
        )

        sp_logits = self.sp_linear(enhanced_sent_nodes)

        gnn_context = torch.matmul(sent_all_mapping, enhanced_sent_nodes)
        fused_state = input_state + gnn_context

        start_logits = self.start_linear(fused_state).squeeze(2) - 1e30 * (
            1 - context_mask
        )
        end_logits = self.end_linear(fused_state).squeeze(2) - 1e30 * (1 - context_mask)

        type_state = torch.max(fused_state, dim=1)[0]
        type_logits = self.type_linear(type_state)

        outer = start_logits[:, :, None] + end_logits[:, None]
        outer_mask = self.get_output_mask(outer)
        outer = outer - 1e30 * (1 - outer_mask[None].expand_as(outer))

        if query_mapping is not None:
            outer = outer - 1e30 * query_mapping[:, :, None]

        start_position = outer.max(dim=2)[0].max(dim=1)[1]
        end_position = outer.max(dim=1)[0].max(dim=1)[1]

        return (
            start_logits,
            end_logits,
            type_logits,
            sp_logits.squeeze(2),
            start_position,
            end_position,
        )


class SupportNet(nn.Module):
    def __init__(self, input_dim, label_type_num, max_answer_len=MAX_ANSWER_SPAN_LEN):
        super(SupportNet, self).__init__()
        self.prediction_layer = SimplePredictionLayer(
            input_dim, label_type_num, max_answer_len=max_answer_len
        )

    def forward(self, batch):
        context_encoding = batch["context_encoding"]
        return self.prediction_layer(batch, context_encoding)


class BertSupportNet(nn.Module):
    def __init__(
        self, encoder, input_dim, label_type_num, max_answer_len=MAX_ANSWER_SPAN_LEN
    ):
        super(BertSupportNet, self).__init__()
        self.encoder = encoder
        self.graph_fusion_net = SupportNet(
            input_dim, label_type_num, max_answer_len=max_answer_len
        )

    def forward(self, batch):
        doc_ids, doc_mask, segment_ids = (
            batch["context_idxs"],
            batch["context_mask"],
            batch["segment_idxs"],
        )
        all_doc_encoder_layers = self.encoder(
            input_ids=doc_ids, token_type_ids=segment_ids, attention_mask=doc_mask
        )[0]
        batch["context_encoding"] = all_doc_encoder_layers
        return self.graph_fusion_net(batch)


def get_final_text(pred_text, orig_text, do_lower_case, verbose_logging=False):
    def _strip_spaces(text):
        ns_chars = []
        ns_to_s_map = collections.OrderedDict()
        for i, c in enumerate(text):
            if c == " ":
                continue
            ns_to_s_map[len(ns_chars)] = i
            ns_chars.append(c)
        ns_text = "".join(ns_chars)
        return (ns_text, ns_to_s_map)

    tokenizer = BasicTokenizer(do_lower_case=do_lower_case)
    tok_text = " ".join(tokenizer.tokenize(orig_text))

    start_position = tok_text.find(pred_text)
    if start_position == -1:
        return orig_text
    end_position = start_position + len(pred_text) - 1

    (orig_ns_text, orig_ns_to_s_map) = _strip_spaces(orig_text)
    (tok_ns_text, tok_ns_to_s_map) = _strip_spaces(tok_text)

    if len(orig_ns_text) != len(tok_ns_text):
        return orig_text

    tok_s_to_ns_map = {}
    for i, tok_index in tok_ns_to_s_map.items():
        tok_s_to_ns_map[tok_index] = i

    orig_start_position = None
    if start_position in tok_s_to_ns_map:
        ns_start_position = tok_s_to_ns_map[start_position]
        if ns_start_position in orig_ns_to_s_map:
            orig_start_position = orig_ns_to_s_map[ns_start_position]
    if orig_start_position is None:
        return orig_text

    orig_end_position = None
    if end_position in tok_s_to_ns_map:
        ns_end_position = tok_s_to_ns_map[end_position]
        if ns_end_position in orig_ns_to_s_map:
            orig_end_position = orig_ns_to_s_map[ns_end_position]
    if orig_end_position is None:
        return orig_text

    return orig_text[orig_start_position : (orig_end_position + 1)]


def reconstruct_span_answer(example_obj, feature_obj, start_pos: int, end_pos: int):
    doc_tokens = feature_obj.doc_tokens
    tok_to_orig_map = feature_obj.token_to_orig_map
    if (
        start_pos < 0
        or end_pos < 0
        or start_pos >= len(doc_tokens)
        or end_pos >= len(doc_tokens)
    ):
        return ""
    if end_pos >= len(tok_to_orig_map):
        return ""

    tok_tokens = doc_tokens[start_pos : end_pos + 1]
    orig_doc_start = tok_to_orig_map[start_pos]
    orig_doc_end = tok_to_orig_map[end_pos]
    if orig_doc_start < 0 or orig_doc_end < 0:
        return ""

    orig_tokens = example_obj.doc_tokens[orig_doc_start : (orig_doc_end + 1)]
    tok_text = " ".join(tok_tokens).replace(" ##", "").replace("##", "").strip()
    tok_text = " ".join(tok_text.split())
    orig_text = " ".join(orig_tokens).strip("[,.;]")
    return get_final_text(
        tok_text, orig_text, do_lower_case=False, verbose_logging=False
    )


class Evaluator:
    @staticmethod
    def normalize_answer(s):
        def remove_articles(text):
            return re.sub(r"\b(a|an|the)\b", " ", text)

        def white_space_fix(text):
            return " ".join(text.split())

        def remove_punc(text):
            exclude = set(string.punctuation)
            return "".join(ch for ch in text if ch not in exclude)

        def lower(text):
            return text.lower()

        return white_space_fix(remove_articles(remove_punc(lower(s))))

    @staticmethod
    def f1_score(prediction, ground_truth):
        normalized_prediction = Evaluator.normalize_answer(prediction)
        normalized_ground_truth = Evaluator.normalize_answer(ground_truth)

        ZERO_METRIC = (0, 0, 0)

        if (
            normalized_prediction in ["yes", "no", "unknown"]
            and normalized_prediction != normalized_ground_truth
        ):
            return ZERO_METRIC
        if (
            normalized_ground_truth in ["yes", "no", "unknown"]
            and normalized_prediction != normalized_ground_truth
        ):
            return ZERO_METRIC

        prediction_tokens = list(normalized_prediction)
        ground_truth_tokens = list(normalized_ground_truth)
        common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
        num_same = sum(common.values())
        if num_same == 0:
            return ZERO_METRIC
        precision = 1.0 * num_same / len(prediction_tokens)
        recall = 1.0 * num_same / len(ground_truth_tokens)
        f1 = (2 * precision * recall) / (precision + recall)
        return f1, precision, recall

    @staticmethod
    def exact_match_score(prediction, ground_truth):
        return Evaluator.normalize_answer(prediction) == Evaluator.normalize_answer(
            ground_truth
        )

    @staticmethod
    def update_answer(metrics, prediction, gold):
        em = Evaluator.exact_match_score(prediction, gold)
        f1, prec, recall = Evaluator.f1_score(prediction, gold)
        metrics["em"] += float(em)
        metrics["f1"] += f1
        metrics["prec"] += prec
        metrics["recall"] += recall
        return em, prec, recall

    @staticmethod
    def update_sp(metrics, prediction, gold):
        cur_sp_pred = set(map(tuple, prediction))
        gold_sp_pred = set(map(tuple, gold))
        tp, fp, fn = 0, 0, 0
        for e in cur_sp_pred:
            if e in gold_sp_pred:
                tp += 1
            else:
                fp += 1
        for e in gold_sp_pred:
            if e not in cur_sp_pred:
                fn += 1
        prec = 1.0 * tp / (tp + fp) if tp + fp > 0 else 0.0
        recall = 1.0 * tp / (tp + fn) if tp + fn > 0 else 0.0
        f1 = 2 * prec * recall / (prec + recall) if prec + recall > 0 else 0.0
        if fp + fn == 0:
            em, prec, recall, f1 = 1.0, 1.0, 1.0, 1.0
        else:
            em = 0.0
        metrics["sp_em"] += em
        metrics["sp_f1"] += f1
        metrics["sp_prec"] += prec
        metrics["sp_recall"] += recall
        return em, prec, recall


def run_eval_from_objects(prediction, gold):
    metrics = {
        "em": 0,
        "f1": 0,
        "prec": 0,
        "recall": 0,
        "sp_em": 0,
        "sp_f1": 0,
        "sp_prec": 0,
        "sp_recall": 0,
        "joint_em": 0,
        "joint_f1": 0,
        "joint_prec": 0,
        "joint_recall": 0,
    }

    for dp in gold:
        cur_id = str(dp["_id"])
        can_eval_joint = True

        if cur_id not in prediction["answer"]:
            can_eval_joint = False
        else:
            em, prec, recall = Evaluator.update_answer(
                metrics, prediction["answer"][cur_id], dp["answer"]
            )

        if cur_id not in prediction["sp"]:
            can_eval_joint = False
        else:
            sp_em, sp_prec, sp_recall = Evaluator.update_sp(
                metrics, prediction["sp"][cur_id], dp["supporting_facts"]
            )

        if can_eval_joint:
            joint_prec = prec * sp_prec
            joint_recall = recall * sp_recall
            if joint_prec + joint_recall > 0:
                joint_f1 = 2 * joint_prec * joint_recall / (joint_prec + joint_recall)
            else:
                joint_f1 = 0.0
            joint_em = em * sp_em

            metrics["joint_em"] += joint_em
            metrics["joint_f1"] += joint_f1
            metrics["joint_prec"] += joint_prec
            metrics["joint_recall"] += joint_recall

    N = len(gold)
    for k in metrics.keys():
        metrics[k] /= N
    return metrics


def compute_loss(
    batch,
    start_logits,
    end_logits,
    type_logits,
    sp_logits,
    criterion_span_sum,
    criterion_type,
    sp_loss_fct,
):
    y1_safe = batch["y1"].clone()
    y2_safe = batch["y2"].clone()
    seq_len = start_logits.size(1)

    y1_safe = torch.where(
        y1_safe >= seq_len, torch.tensor(IGNORE_INDEX, device=y1_safe.device), y1_safe
    )
    y2_safe = torch.where(
        y2_safe >= seq_len, torch.tensor(IGNORE_INDEX, device=y2_safe.device), y2_safe
    )

    start_count = (y1_safe != IGNORE_INDEX).sum().clamp_min(1)
    end_count = (y2_safe != IGNORE_INDEX).sum().clamp_min(1)

    loss_span = (
        criterion_span_sum(start_logits, y1_safe) / start_count
        + criterion_span_sum(end_logits, y2_safe) / end_count
    )
    loss_type = TYPE_LOSS_WEIGHT * criterion_type(type_logits, batch["q_type"])

    sent_num_in_batch = batch["sent_start_mapping"].sum().clamp_min(1.0)
    sp_loss_raw = sp_loss_fct(sp_logits.view(-1), batch["is_support"].float().view(-1))
    loss_sp = SP_LOSS_WEIGHT * sp_loss_raw.sum() / (sent_num_in_batch + 1e-9)

    total_loss = loss_span + loss_type + loss_sp
    return total_loss, loss_span, loss_type, loss_sp


@torch.no_grad()
def collect_dev_outputs(model, dataloader, example_dict):
    model.eval()
    dataloader.refresh()

    type_max = {}
    best_span = {}
    sp_score_maps = {}

    for batch in tqdm(dataloader, desc="Evaluating", **EVAL_TQDM_KW):
        batch["context_mask"] = batch["context_mask"].float()
        start_logits, end_logits, type_logits, sp_logits, s_pos, e_pos = model(batch)

        feats = batch["features"]
        ids = batch["ids"]

        type_np = type_logits.detach().cpu().numpy()
        start_np = start_logits.detach().cpu().numpy()
        end_np = end_logits.detach().cpu().numpy()
        prob_np = torch.sigmoid(sp_logits).detach().cpu().numpy()
        s_pos_np = s_pos.detach().cpu().numpy()
        e_pos_np = e_pos.detach().cpu().numpy()

        for i in range(len(ids)):
            qid_int = ids[i]
            qid = str(qid_int)
            feat = feats[i]

            cur_type = type_np[i]
            if qid not in type_max:
                type_max[qid] = cur_type.copy()
            else:
                type_max[qid] = np.maximum(type_max[qid], cur_type)

            s_i = int(s_pos_np[i])
            e_i = int(e_pos_np[i])
            if 0 <= s_i < start_np.shape[1] and 0 <= e_i < end_np.shape[1]:
                span_score = float(start_np[i, s_i] + end_np[i, e_i])
                if (qid not in best_span) or (span_score > best_span[qid][0]):
                    best_span[qid] = (span_score, feat, s_i, e_i)

            if qid not in sp_score_maps:
                sp_score_maps[qid] = {}
            L = min(prob_np.shape[1], len(feat.sent_names))
            for j in range(L):
                name = feat.sent_names[j]
                p = float(prob_np[i, j])
                old = sp_score_maps[qid].get(name)
                if old is None or p > old:
                    sp_score_maps[qid][name] = p

    answer_dict = {}
    for qid, tvec in type_max.items():
        pred_type = int(np.argmax(tvec))
        if pred_type == 1:
            answer_dict[qid] = "yes"
        elif pred_type == 2:
            answer_dict[qid] = "no"
        elif pred_type == 3:
            answer_dict[qid] = "unknown"
        else:
            qid_int = int(qid) if qid.isdigit() else qid
            ex = example_dict[qid_int]
            if qid in best_span:
                _, feat, s_i, e_i = best_span[qid]
                ans = reconstruct_span_answer(ex, feat, s_i, e_i)
                answer_dict[qid] = ans.replace(" ", "")
            else:
                answer_dict[qid] = ""

    model.train()
    return answer_dict, sp_score_maps


def build_prediction_from_scores(answer_dict, sp_score_maps, threshold: float):
    thr = float(threshold)
    sp_dict = {}
    for qid, m in sp_score_maps.items():
        sp_dict[qid] = [name for (name, p) in m.items() if p > thr]
    return {"answer": answer_dict, "sp": sp_dict}


def find_best_threshold(answer_dict, sp_score_maps, gold, metric_key: str):
    best_thr = SP_PROB_THRESHOLD
    best_metrics = None
    best_score = -1e18

    for thr in SP_THRESHOLD_COARSE_GRID:
        pred = build_prediction_from_scores(answer_dict, sp_score_maps, thr)
        metrics = run_eval_from_objects(pred, gold)
        score = float(metrics[metric_key])
        if score > best_score:
            best_score = score
            best_thr = float(thr)
            best_metrics = metrics

    lo = max(0.0, best_thr - SP_THRESHOLD_FINE_RADIUS)
    hi = min(1.0, best_thr + SP_THRESHOLD_FINE_RADIUS)
    fine = np.arange(lo, hi + 1e-12, SP_THRESHOLD_FINE_STEP)

    for thr in fine:
        pred = build_prediction_from_scores(answer_dict, sp_score_maps, thr)
        metrics = run_eval_from_objects(pred, gold)
        score = float(metrics[metric_key])
        if score > best_score:
            best_score = score
            best_thr = float(thr)
            best_metrics = metrics

    return best_thr, best_metrics


def main():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is required (this script builds CUDA tensors in the iterator)."
        )

    torch.cuda.set_device(CUDA_DEVICE_ID)
    set_seed(RANDOM_SEED)

    os.makedirs(PROCESSED_DATA_ROOT, exist_ok=True)
    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    ensure_preprocessed()

    with open(DEV_DATA_PATH, "r", encoding="utf-8") as f:
        dev_gold = json.load(f)

    data_helper = DataHelper(PROCESSED_DATA_ROOT, gz=True)
    train_loader = data_helper.train_loader(
        bsz=TRAIN_BATCH_SIZE, device=CUDA_DEVICE_ID, sent_limit=MAX_SENTENCES
    )
    dev_loader = data_helper.dev_loader(
        bsz=EVAL_BATCH_SIZE, device=CUDA_DEVICE_ID, sent_limit=MAX_SENTENCES
    )

    local_only = _local_only(PRETRAINED_MODEL_PATH)
    config, encoder = _load_config_and_encoder(
        PRETRAINED_MODEL_PATH, local_only=local_only
    )

    model = BertSupportNet(
        encoder=encoder,
        input_dim=config.hidden_size,
        label_type_num=NUM_ANSWER_TYPES,
        max_answer_len=MAX_ANSWER_SPAN_LEN,
    ).cuda()

    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)

    t_total = len(train_loader) * NUM_EPOCHS // GRAD_ACCUM_STEPS
    warmup_steps = int(0.1 * t_total)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=t_total
    )
    last_lr = optimizer.param_groups[0]["lr"]

    scaler = _grad_scaler(USE_AMP)
    criterion_span_sum = nn.CrossEntropyLoss(reduction="sum", ignore_index=IGNORE_INDEX)
    criterion_type = nn.CrossEntropyLoss(reduction="mean")
    sp_loss_fct = nn.BCEWithLogitsLoss(reduction="none")

    checkpoint_manager = CheckpointManager(OUTPUT_ROOT, EXPERIMENT_NAME)
    early_stopper = (
        EarlyStopping(
            patience=EARLY_STOP_PATIENCE,
            mode=EARLY_STOP_MODE,
            min_delta=EARLY_STOP_MIN_DELTA,
        )
        if ENABLE_EARLY_STOPPING
        else None
    )

    global_step = 0
    best_epoch = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loader.refresh()
        model.train()

        running_loss = [0.0, 0.0, 0.0, 0.0]
        running_steps = 0
        nan_steps = 0

        pbar = tqdm(total=len(train_loader), desc=f"Epoch {epoch}", **TRAIN_TQDM_KW)

        data_iter = iter(train_loader)
        while not train_loader.empty():
            try:
                batch = next(data_iter)
            except StopIteration:
                data_iter = iter(train_loader)
                batch = next(data_iter)

            batch["context_mask"] = batch["context_mask"].float()

            with _autocast(USE_AMP):
                start_logits, end_logits, type_logits, sp_logits, _, _ = model(batch)
                loss, loss_span, loss_type, loss_sp = compute_loss(
                    batch,
                    start_logits,
                    end_logits,
                    type_logits,
                    sp_logits,
                    criterion_span_sum,
                    criterion_type,
                    sp_loss_fct,
                )
                loss_for_backward = (
                    loss / GRAD_ACCUM_STEPS if GRAD_ACCUM_STEPS > 1 else loss
                )

            if not torch.isfinite(loss_for_backward):
                optimizer.zero_grad(set_to_none=True)
                nan_steps += 1
                global_step += 1
                pbar.update(1)
                continue

            if USE_AMP and scaler is not None:
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            if (global_step + 1) % GRAD_ACCUM_STEPS == 0:
                if USE_AMP and scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                last_lr = scheduler.get_last_lr()[0]
                optimizer.zero_grad(set_to_none=True)

            running_loss[0] += float(loss.detach().cpu().item())
            running_loss[1] += float(loss_span.detach().cpu().item())
            running_loss[2] += float(loss_type.detach().cpu().item())
            running_loss[3] += float(loss_sp.detach().cpu().item())
            running_steps += 1
            global_step += 1

            denom = float(max(running_steps, 1))
            pbar.set_postfix(
                {
                    "loss": f"{running_loss[0] / denom:.4f}",
                    "span": f"{running_loss[1] / denom:.4f}",
                    "type": f"{running_loss[2] / denom:.4f}",
                    "sp": f"{running_loss[3] / denom:.4f}",
                    "lr": f"{last_lr:.2e}",
                    "nan": nan_steps,
                },
                refresh=False,
            )
            pbar.update(1)

        pbar.close()

        model_to_save = model.module if hasattr(model, "module") else model
        checkpoint_manager.save_last_ckpt(model_to_save.state_dict())

        answer_dict, sp_score_maps = collect_dev_outputs(
            model, dev_loader, data_helper.dev_example_dict
        )

        if ENABLE_SP_THRESHOLD_SEARCH:
            best_thr, metrics = find_best_threshold(
                answer_dict, sp_score_maps, dev_gold, EARLY_STOP_METRIC
            )
        else:
            best_thr = SP_PROB_THRESHOLD
            pred_obj = build_prediction_from_scores(
                answer_dict, sp_score_maps, best_thr
            )
            metrics = run_eval_from_objects(pred_obj, dev_gold)

        pred_obj = build_prediction_from_scores(answer_dict, sp_score_maps, best_thr)
        _atomic_write_json(checkpoint_manager.tmp_pred_path, pred_obj)
        checkpoint_manager.finalize_last_pred_from_tmp()

        checkpoint_manager.save_last_metrics(
            {"epoch": epoch, "sp_threshold": best_thr, "metrics": metrics}
        )
        _tqdm_write(json.dumps({"sp_threshold": best_thr}, ensure_ascii=False))
        _tqdm_write(json.dumps(metrics, ensure_ascii=False, indent=4))

        if ENABLE_EARLY_STOPPING:
            if EARLY_STOP_METRIC not in metrics:
                raise KeyError(
                    f"EARLY_STOP_METRIC='{EARLY_STOP_METRIC}' not found in metrics: {list(metrics.keys())}"
                )

            current_score = float(metrics[EARLY_STOP_METRIC])
            improved, should_stop = early_stopper.step(current_score)

            if improved:
                best_epoch = epoch
                checkpoint_manager.save_best_ckpt_from_last()
                checkpoint_manager.save_best_pred_from_last()
                checkpoint_manager.save_best_metrics()

            _tqdm_write(
                json.dumps(
                    {
                        "epoch": epoch,
                        "best_epoch": best_epoch,
                        "best_score": early_stopper.best_score,
                        "bad_epochs": early_stopper.num_bad_epochs,
                    },
                    ensure_ascii=False,
                )
            )

            if should_stop:
                _tqdm_write(
                    json.dumps(
                        {
                            "early_stop": True,
                            "stop_epoch": epoch,
                            "best_epoch": best_epoch,
                            "best_score": early_stopper.best_score,
                        },
                        ensure_ascii=False,
                    )
                )
                break


if __name__ == "__main__":
    main()